# Azure Key Vault

## 1. What is Azure Key Vault?

**Azure Key Vault** is a managed Azure service used to securely store and manage sensitive information such as:

- Secrets
- Encryption keys
- Certificates

Instead of putting credentials directly in application code:

```python
AZURE_API_KEY = "abc123"
DB_PASSWORD = "password"
```

store them in Key Vault:

```text
Application
     ↓
Microsoft Entra ID
     ↓
Managed Identity
     ↓
Azure Key Vault
     ↓
Secret / Key / Certificate
```

---

# 2. Why Do We Need Key Vault?

Without Key Vault:

```text
Python
 ├── OpenAI API Key
 ├── DB Password
 ├── Storage Key
 └── Third-party API Secret
```

Problems:

- Secrets can accidentally enter Git
- Difficult to rotate credentials
- Developers may have direct access to production secrets
- Secret management becomes distributed across applications

With Key Vault:

```text
Python Application
       ↓
Managed Identity
       ↓
Key Vault
       ↓
Get Secret
```

The application doesn't need to hard-code the secret.

---

# 3. What Can Key Vault Store?

| Object | Purpose | Example |
|---|---|---|
| **Secrets** | Sensitive values | API keys, passwords, connection strings |
| **Keys** | Cryptographic keys | Encryption/decryption |
| **Certificates** | TLS/SSL certificates | HTTPS certificates |

### Simple distinction

```text
Secret      → "What is the password?"
Key         → "How do I encrypt/decrypt?"
Certificate → "How do I prove identity securely?"
```

---

# 4. Key Vault + Managed Identity ⭐⭐⭐⭐⭐

This is the most important architecture to understand.

Suppose an Azure Function needs a database password.

### Don't do this:

```python
db_password = "MyProductionPassword"
```

Instead:

```text
Azure Function
      │
      ▼
Managed Identity
      │
      ▼
Microsoft Entra ID
      │
      ▼
Azure Key Vault
      │
      ▼
DB Password
```

The Function authenticates to Key Vault using its managed identity.

---

# 5. Complete Authentication Flow

```text
1. Azure Function starts
          ↓
2. Function has Managed Identity
          ↓
3. Managed Identity exists in Entra ID
          ↓
4. Function requests access token
          ↓
5. Entra ID issues token
          ↓
6. Function calls Key Vault
          ↓
7. Key Vault checks permissions
          ↓
8. Secret returned
```

So:

**Entra ID → authentication**

**RBAC → authorization**

**Key Vault → secret management**

---

# 6. Python Example

Install:

```bash
pip install azure-identity azure-keyvault-secrets
```

Then:

```python
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

vault_url = "https://my-key-vault.vault.azure.net/"

credential = DefaultAzureCredential()

client = SecretClient(
    vault_url=vault_url,
    credential=credential
)

secret = client.get_secret("OpenAI-Api-Key")

print(secret.value)
```

Notice that there is no Key Vault password in the code.

```text
DefaultAzureCredential
        ↓
Managed Identity in Azure
        ↓
Entra ID
        ↓
Key Vault
```

---

# 7. `DefaultAzureCredential`

This is very useful for development and production.

```python
credential = DefaultAzureCredential()
```

Locally, it can use developer authentication such as Azure CLI credentials.

In Azure, it can use the application's managed identity.

```text
LOCAL
────────────────
Developer
   ↓
DefaultAzureCredential
   ↓
Azure CLI / Developer Identity


PRODUCTION
────────────────
Azure App / Function
   ↓
DefaultAzureCredential
   ↓
Managed Identity
   ↓
Entra ID
```

This allows you to avoid changing authentication code between environments.

---

# 8. Key Vault + Azure OpenAI

Suppose you have an Azure OpenAI endpoint and API key.

### Less preferred:

```text
Environment Variable
       ↓
AZURE_OPENAI_API_KEY
       ↓
Application
```

### Enterprise pattern:

```text
AI Application
      ↓
Managed Identity
      ↓
Key Vault
      ↓
Azure OpenAI Credential
```

However, when Azure OpenAI supports **Microsoft Entra ID / managed identity authentication**, you should generally prefer that over retrieving an Azure OpenAI API key from Key Vault.

This is an important senior-level distinction:

> **Key Vault is not automatically required for every Azure credential. If the target service supports Entra ID authentication, passwordless authentication is generally preferable.**

---

# 9. Key Vault + RAG

Consider your Azure RAG system:

```text
                    AI Application
                         │
                  Managed Identity
                         │
        ┌────────────────┼─────────────────┐
        ▼                ▼                 ▼
    Key Vault       Azure AI Search    Azure OpenAI
        │
        ▼
 Third-party secrets
```

Key Vault can manage credentials that your application genuinely needs.

For example:

- External API key
- Database credentials
- Third-party service secret

While Azure-native services should preferably use Managed Identity where supported.

---

# 10. Key Vault + Agentic AI

Suppose your agent has:

```text
HR Agent
   │
   ├── Azure OpenAI
   ├── Azure AI Search
   ├── HR API
   ├── Payroll API
   └── External Service
```

The agent shouldn't have secrets embedded in its prompt or source code.

```text
Agent
  │
  ├── Managed Identity
  │
  ▼
Key Vault
  │
  ├── HR API Secret
  ├── Payroll API Secret
  └── External API Secret
```

For Azure-native resources, use managed identity directly wherever possible.

---

# 11. Key Vault vs Managed Identity

This is a very common interview question.

They solve **different problems**.

| Managed Identity | Azure Key Vault |
|---|---|
| Provides workload identity | Stores/manages secrets and keys |
| Authentication | Secret management |
| Avoids application credentials | Securely stores credentials when needed |
| Integrated with Entra ID | Protected by Entra ID/RBAC |
| Doesn't store your API keys | Stores secrets |
| "Who am I?" | "Where is the secret?" |

### Together

```text
Managed Identity
       ↓
"Who am I?"
       ↓
Entra ID
       ↓
Authorization
       ↓
Key Vault
       ↓
"Give me the secret."
```

---

# 12. Key Vault vs Environment Variables

| Environment Variables | Key Vault |
|---|---|
| Simple | Enterprise secret management |
| Secrets still exist in app environment | Centralized management |
| Rotation is application-dependent | Supports secret lifecycle management |
| Risk of accidental exposure | Better access control/auditing |
| Good for local development | Better for production |

Environment variables aren't inherently insecure, but **centralized secret management is generally preferable for production workloads**.

---

# 13. Secret Rotation

Suppose:

```text
Third-party API Key
     ↓
Key Vault
```

The key changes.

Instead of changing code:

```text
❌ Deploy application
❌ Change source code
❌ Update Git
```

you can update the secret in Key Vault.

```text
Key Vault
   ↓
New Secret Version
   ↓
Application retrieves current version
```

Key Vault supports versioning of secrets.

---

# 14. Secret Versioning

Conceptually:

```text
OpenAI-Key

Version 1 → Old
Version 2 → Current
Version 3 → New
```

Applications can retrieve a specific version or the current version depending on how they access the secret.

This is useful for controlled rotation and rollback scenarios.

---

# 15. RBAC and Access Control

You should follow **least privilege**.

Example:

```text
AI Application Identity
       ↓
Key Vault
       ↓
Secret Permissions
       ↓
Only required secrets
```

Don't give every developer:

```text
Full Key Vault Administrator
```

Instead, define appropriate roles and scopes.

---

# 16. Network Security

For higher-security environments, Key Vault can be integrated with Azure networking controls such as:

- Private endpoints
- Virtual networks
- Firewall/network access controls

Conceptually:

```text
AI Application
      │
 Private Network
      │
      ▼
Private Endpoint
      │
      ▼
Azure Key Vault
```

This helps keep sensitive traffic off the public internet where the architecture requires private connectivity.

---

# 17. Auditing

For production applications, you should know:

```text
Who accessed?
What was accessed?
When?
From where?
Was access successful?
```

Key Vault integrates with Azure monitoring/logging capabilities for auditing and operational visibility.

---

# 18. Key Vault in Production AI Architecture

Combine everything you've studied:

```text
                         USER
                           │
                           ▼
                    Microsoft Entra ID
                           │
                           ▼
                     AI Application
                           │
                    Managed Identity
                           │
         ┌─────────────────┼──────────────────┐
         │                 │                  │
         ▼                 ▼                  ▼
    Azure OpenAI     Azure AI Search      Key Vault
         │                 │                  │
         │                 │             Secrets
         │                 │
         └────────┬────────┘
                  ▼
              RAG / Agent
                  │
                  ▼
           Content Safety
                  │
                  ▼
               Response
```

---

# 19. Key Vault in an Agentic Workflow

Example:

> Agent needs to call an external payroll API.

```text
User
 ↓
Agent
 ↓
Tool Call
 ↓
Payroll API Client
 ↓
Managed Identity
 ↓
Key Vault
 ↓
Payroll API Credential
 ↓
Payroll API
 ↓
Result
 ↓
Agent
 ↓
User
```

But if the payroll API supports Entra ID/OAuth directly, prefer that authentication mechanism instead of storing a static API secret.

---

# 20. Common Interview Questions

### Q1. What is Azure Key Vault?

> "Azure Key Vault is a managed Azure service for securely storing and managing secrets, cryptographic keys, and certificates."

### Q2. Why use Key Vault?

> "It centralizes sensitive information, provides controlled access and auditing, supports secret lifecycle management and reduces the need to hard-code credentials in applications."

### Q3. How does Managed Identity work with Key Vault?

> "The Azure application uses its managed identity to obtain an Entra ID access token. Key Vault validates that identity and its assigned permissions before returning the requested secret."

### Q4. Managed Identity vs Key Vault?

> "Managed Identity provides the workload identity and authentication mechanism, whereas Key Vault securely stores secrets and cryptographic material. They are complementary."

### Q5. Would you store an Azure OpenAI API key in Key Vault?

> "If API-key authentication is required, Key Vault is an appropriate place to store the key. However, if the Azure service supports Microsoft Entra ID and managed identity authentication, I would generally prefer passwordless authentication and avoid a static API key altogether."

### Q6. How would you secure Key Vault?

> "I would use Entra ID with RBAC, managed identities, least-privilege access, private endpoints/network restrictions where required, secret rotation/versioning, and monitoring/auditing."

---

# 21. Senior-Level Scenario

### Interviewer:

> "You have a Python-based Azure Agentic AI application deployed to App Service. It needs Azure AI Search, Azure OpenAI, Blob Storage and a third-party API. How would you manage credentials?"

### Strong answer:

> "I would enable a managed identity on the App Service and use `DefaultAzureCredential` for Azure-native services wherever Entra ID authentication is supported. I would assign least-privilege RBAC permissions to Azure AI Search and Blob Storage. For the third-party API, if it doesn't support Entra ID, I would store its API credential in Azure Key Vault and allow the application's managed identity to retrieve it. I would also apply network restrictions, auditing, and secret rotation."

That is the **production-grade answer** interviewers generally want to hear.